# Итоговое исследование по ОЕЯ — тематическая классификация новостей
Сравниваются **классика (TF-IDF + LogisticRegression, MultinomialNB, LinearSVC)**, **TextCNN на PyTorch** и **RuBERT-tiny2 с fine-tuning классификационной головы и энкодера**.

Язык русский. Темы — образование, развлечения, технологии. Модель получает текст после ASR. Этот ноутбук закрывает исследовательскую часть задания. ASR и сетевой Telegram-бот находятся в общем репозитории.

### Как запустить
1. Клонировать [news_classifier_bot](https://github.com/TaurineMerge/news_classifier_bot), установить приложенный requirements-nlp.txt.
2. Распаковать архив данных из [папки data](https://drive.google.com/drive/folders/1xd6hAMbY1ZT-MbCYGsvA5-hTeh6BM5SA) в ml/data/, чтобы появилась lenta_v3.
3. Указать REPO_ROOT в следующей ячейке. В Colab можно использовать /content/news_classifier_bot и включить GPU.
4. Выполнить все ячейки. RETRAIN=True действительно обучает все модели. RETRAIN=False загружает нейросетевые чекпоинты этого ноутбука, классика обучается заново.
5. Для полной подготовки исходного корпуса включить REBUILD_DATA=True — скачивается закреплённый архив и выполняется подготовка.

Результаты относятся к **предварительным автоматическим меткам**, построенным по рубрикам, тегам и заголовкам Lenta. Качество на естественных голосовых отдельно не измеряется.

In [ ]:
# ruff: noqa: I001
import os
from pathlib import Path

REPO_ROOT = Path(os.environ.get("NEWSBOT_REPO", Path.cwd())).resolve()
if not (REPO_ROOT / "ml/scripts/lenta_data.py").is_file():
    REPO_ROOT = next(
        (
            p / "work/news_classifier_bot"
            for p in [REPO_ROOT, *REPO_ROOT.parents]
            if (p / "work/news_classifier_bot/ml/scripts/lenta_data.py").exists()
        ),
        REPO_ROOT,
    )
assert (REPO_ROOT / "ml/scripts/lenta_data.py").is_file(), "Укажите REPO_ROOT"
RETRAIN = True
REBUILD_DATA = False
RUN_DIR = Path(os.environ.get("NEWSBOT_NLP_RUN", REPO_ROOT / "ml/models/nlp_final")).resolve()
BASE_DIR = Path(os.environ.get("NEWSBOT_RUBERT_BASE", RUN_DIR / "rubert-base"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = REPO_ROOT / "ml/data/lenta_v3"
print("Репозиторий", REPO_ROOT)
print("Артефакты", RUN_DIR)

In [ ]:
import copy
import hashlib
import json
import random
import re
import sys
from collections import Counter
from time import perf_counter
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    average_precision_score,
    precision_recall_curve,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

sys.path.insert(0, str(REPO_ROOT))
from ml.scripts.download_lenta import download
from ml.scripts.lenta_data import prepare
from ml.scripts.train_text_baseline import read_splits

torch.set_num_threads(2)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
LABELS = ["education", "entertainment", "technology"]
RUSSIAN = ["образование", "развлечения", "технологии"]
VERSIONS = dict(
    python=sys.version.split()[0],
    sklearn=sklearn.__version__,
    torch=torch.__version__,
    transformers=transformers.__version__,
    numpy=np.__version__,
)
print(DEVICE, VERSIONS)

## 1. Подготовка данных
Источник Lenta v1.1 — 800 975 строк. Все три темы получены из этого корпуса по правилам и редакционным признакам. Заголовок участвует в присвоении метки, а вход модели — первые 140 слов **тела** статьи.

prepare() проверяет SHA-256 исходного архива, исключает некорректные даты, пустые идентификаторы, тексты короче 30 слов и точные дубли. Для дублей используются нижний регистр, ё→е и схлопывание пробелов. Отбор с seed=20261028 балансирует темы. Train — 1999–2017, Validation — 2018, Test — 2019.

read_splits() повторно проверяет хеши частей, набор классов, даты и отсутствие точных пересечений по ID, URL, заголовку и входу. Семантически близкие новости об одном событии отдельно не группировались.

In [ ]:
if REBUILD_DATA:
    prepare(download(REPO_ROOT / "ml/data/lenta-ru-news.csv.bz2"), DATA_DIR)
splits, metadata = read_splits(DATA_DIR)
texts = {k: [r["text"] for r in rows] for k, rows in splits.items()}
targets = {k: np.array([LABELS.index(r["label"]) for r in rows]) for k, rows in splits.items()}
hashes = {k: hashlib.sha256((DATA_DIR / f"{k}.jsonl").read_bytes()).hexdigest() for k in splits}
display(
    pd.DataFrame(
        [{"часть": k, "статей": len(rows), **Counter(r["label"] for r in rows)} for k, rows in splits.items()]
    )
)
lengths = pd.DataFrame(
    [{"часть": k, "слов": len(r["text"].split())} for k, rows in splits.items() for r in rows]
)
display(lengths.groupby("часть")["слов"].describe().round(1))
lengths.hist(column="слов", by="часть", bins=15, figsize=(10, 3))
plt.tight_layout()
plt.show()

## 2. Метрики и порядок выбора
Accuracy — доля верных ответов. Macro-F1 даёт каждой теме одинаковый вес. Weighted-F1 усредняет F1 с весом по числу примеров. **PR-AUC определяем как macro average precision по one-vs-rest**, без трапециевидного интегрирования. LinearSVC оценивается по decision scores; остальные модели — по вероятностям.

ROC-AUC в ТЗ указан для бинарной классификации. Здесь три класса, поэтому используем многоклассовый PR-AUC и указанный набор метрик.

Параметры и эпохи выбираются по validation macro-F1. Test оценивается после выбора. Исторически этот Test уже использовался в проекте — это фиксированная сравнительная выборка, а не новый закрытый тест. Разбор ошибок и срезы ниже не используются для подбора параметров.

В Test по 122 примера каждого класса, поэтому macro-F1 и weighted-F1 совпадают.

In [ ]:
def metrics(y, scores):
    assert scores.shape == (len(y), 3) and np.isfinite(scores).all()
    pred = scores.argmax(1)
    return dict(
        accuracy=accuracy_score(y, pred),
        macro_f1=f1_score(y, pred, labels=range(3), average="macro", zero_division=0),
        weighted_f1=f1_score(y, pred, labels=range(3), average="weighted", zero_division=0),
        pr_auc_macro_ap_ovr=average_precision_score(np.eye(3)[y], scores, average="macro"),
    )


models, validation, records, histories = {}, {}, {}, {}

## 3. Классические классификаторы
TF-IDF по словам и парам слов, min_df=2, max_df=0.98, до 40 000 признаков, sublinear_tf=True. Векторизатор строится только на Train. Лемматизация и удаление стоп-слов не применяются. Train сбалансирован, class_weight не задаётся.

MultinomialNB — конкретная разновидность наивного Байеса. Alpha=0.5 сглаживает оценки редких признаков. C=1 задаёт регуляризацию логрега и SVM.

In [ ]:
estimators = {
    "LogisticRegression": LogisticRegression(C=1, max_iter=1000, random_state=42),
    "MultinomialNB": MultinomialNB(alpha=0.5),
    "LinearSVC": LinearSVC(C=1, random_state=42),
}


def classic_scores(model, batch):
    scores = model.predict_proba(batch) if hasattr(model, "predict_proba") else model.decision_function(batch)
    return scores[:, [list(model.classes_).index(label) for label in LABELS]]


for name, estimator in estimators.items():
    start = perf_counter()
    model = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=40000,
            sublinear_tf=True,
            dtype=np.float32,
        ),
        estimator,
    )
    model.fit(texts["train"], [LABELS[i] for i in targets["train"]])
    models[name] = model
    records[name] = {
        "fit_seconds": perf_counter() - start,
        "parameters": estimator.get_params(),
    }
    validation[name] = metrics(targets["validation"], classic_scores(model, texts["validation"]))
    print(name, validation[name])

## 4. TextCNN на PyTorch
Словарь до 20 000 токенов, только Train. PAD=0, UNK=1, длина до 256 токенов. Обучаемый Embedding размерности 64, три Conv1d (ширина 3/4/5, 64 фильтра), ReLU, max-pooling, dropout, линейная голова на три класса.

Два запуска: seed=42/dropout=0.3 и seed=20261125/dropout=0.5. AdamW, lr=0.001, weight_decay=0.01, batch=64, 8 эпох. По Validation выбираются эпоха и запуск. Seed и dropout меняются вместе, поэтому причинный эффект dropout отдельно не измеряется. CNN допустима по пункту 2.2 задания.

In [ ]:
def tokens(text):
    return re.findall(r"\b\w\w+\b", text.lower())


counts = Counter(w for t in texts["train"] for w in tokens(t))
vocabulary = {"<PAD>": 0, "<UNK>": 1}
for w, n in sorted(counts.items(), key=lambda p: (-p[1], p[0])):
    if n >= 2 and len(vocabulary) < 20000:
        vocabulary[w] = len(vocabulary)


def cnn_encode(batch):
    ids = torch.zeros((len(batch), 256), dtype=torch.long)
    for i, t in enumerate(batch):
        values = [vocabulary.get(w, 1) for w in tokens(t)[:256]]
        ids[i, : len(values)] = torch.tensor(values, dtype=torch.long)
    return ids


class TextCNN(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
        self.convolutions = nn.ModuleList([nn.Conv1d(64, 64, k) for k in (3, 4, 5)])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(192, 3)

    def forward(self, ids):
        embedding = self.embedding(ids).transpose(1, 2)
        features = [torch.relu(layer(embedding)).amax(2) for layer in self.convolutions]
        return self.classifier(self.dropout(torch.cat(features, 1)))


@torch.inference_mode()
def cnn_scores(model, batch):
    model.eval()
    return np.concatenate(
        [model(ids.to(DEVICE)).softmax(-1).cpu().numpy() for ids in cnn_encode(batch).split(64)]
    )


cnn_runs = []
for seed, dropout in ((42, 0.3), (20261125, 0.5)):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    model = TextCNN(dropout).to(DEVICE)
    path = RUN_DIR / f"cnn_{seed}.pt"
    if RETRAIN:
        loader = DataLoader(
            TensorDataset(cnn_encode(texts["train"]), torch.tensor(targets["train"])),
            batch_size=64,
            shuffle=True,
            generator=torch.Generator().manual_seed(seed),
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        history, best, state, best_epoch = [], -1, None, 0
        for epoch in range(1, 9):
            model.train()
            losses = []
            for ids, y in loader:
                optimizer.zero_grad(set_to_none=True)
                loss = nn.functional.cross_entropy(model(ids.to(DEVICE)), y.to(DEVICE))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1)
                optimizer.step()
                losses.append(loss.item())
            val = metrics(targets["validation"], cnn_scores(model, texts["validation"]))
            history.append(dict(epoch=epoch, loss=float(np.mean(losses)), **val))
            if val["macro_f1"] > best:
                best, best_epoch = val["macro_f1"], epoch
                state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            print("CNN", seed, "epoch", epoch, "val F1", round(val["macro_f1"], 4))
        torch.save(
            dict(
                state=state,
                vocabulary=vocabulary,
                history=history,
                best_epoch=best_epoch,
                data_hashes=hashes,
                dropout=dropout,
            ),
            path,
        )
    saved = torch.load(path, map_location="cpu", weights_only=True)
    assert saved["vocabulary"] == vocabulary and saved["data_hashes"] == hashes
    model.load_state_dict(saved["state"])
    val = metrics(targets["validation"], cnn_scores(model, texts["validation"]))
    cnn_runs.append((val["macro_f1"], seed, model, saved))
best_cnn = max(cnn_runs, key=lambda r: r[0])
models["TextCNN"] = best_cnn[2]
histories["TextCNN"] = best_cnn[3]["history"]
validation["TextCNN"] = metrics(targets["validation"], cnn_scores(best_cnn[2], texts["validation"]))
records["TextCNN"] = {
    "seed": best_cnn[1],
    "epoch": best_cnn[3]["best_epoch"],
    "runs": [
        {"seed": seed, "epoch": s["best_epoch"], "validation_macro_f1": v} for v, seed, _, s in cnn_runs
    ],
}
print("Выбран CNN", records["TextCNN"])

## 5. Fine-tuning RuBERT-tiny2
[cointegrated/rubert-tiny2](https://huggingface.co/cointegrated/rubert-tiny2), закреплённая ревизия e8ed3b0c8bbf4fb6984c3de043bf7d2f4e5969ae. Новая классификационная голова имеет три выхода. Дообучаются **все параметры, включая голову**.

AdamW, lr=5e-5, weight_decay=0.01, batch=32, 3 эпохи, 10% warmup и линейное уменьшение learning rate. WordPiece, до 192 субтокенов; входные тексты те же, но обрезка отличается от TF-IDF/CNN. Лучшая эпоха выбирается по Validation.

In [ ]:
from huggingface_hub import snapshot_download

REVISION = "e8ed3b0c8bbf4fb6984c3de043bf7d2f4e5969ae"
BASE_SHA = "26ebb6db2a68593c54c74902d7a74f332da66297693f965cc9f1b0af4abf3894"
if not (BASE_DIR / "model.safetensors").exists():
    snapshot_download(
        "cointegrated/rubert-tiny2",
        revision=REVISION,
        local_dir=BASE_DIR,
        allow_patterns=[
            "config.json",
            "tokenizer*.json",
            "special_tokens_map.json",
            "vocab.txt",
            "model.safetensors",
        ],
    )
with (BASE_DIR / "model.safetensors").open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == BASE_SHA
tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, local_files_only=True)
torch.manual_seed(20261106)
random.seed(20261106)
np.random.seed(20261106)
bert_path = RUN_DIR / "rubert"
bert = AutoModelForSequenceClassification.from_pretrained(
    BASE_DIR if RETRAIN else bert_path,
    local_files_only=True,
    use_safetensors=True,
    num_labels=3,
    id2label=dict(enumerate(LABELS)),
    label2id={label: i for i, label in enumerate(LABELS)},
    attn_implementation="eager",
).to(DEVICE)


@torch.inference_mode()
def bert_scores(model, batch):
    model.eval()
    results = []
    for start in range(0, len(batch), 32):
        encoded = tokenizer(
            batch[start : start + 32],
            padding=True,
            truncation=True,
            max_length=192,
            return_tensors="pt",
        ).to(DEVICE)
        values = model(**encoded).logits.softmax(-1).cpu().numpy()
        results.append(values[:, [model.config.label2id[label] for label in LABELS]])
    return np.concatenate(results)


if RETRAIN:

    def collate(rows):
        batch = tokenizer(
            [r["text"] for r in rows],
            padding=True,
            truncation=True,
            max_length=192,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor([LABELS.index(r["label"]) for r in rows])
        return batch

    loader = DataLoader(
        splits["train"],
        batch_size=32,
        shuffle=True,
        collate_fn=collate,
        generator=torch.Generator().manual_seed(20261106),
    )
    optimizer = torch.optim.AdamW(bert.parameters(), lr=5e-5, weight_decay=0.01)
    total_steps = 3 * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
    history, best, state, best_epoch = [], -1, None, 0
    for epoch in range(1, 4):
        bert.train()
        losses = []
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = bert(**batch.to(DEVICE)).loss
            loss.backward()
            nn.utils.clip_grad_norm_(bert.parameters(), 1)
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
        val = metrics(targets["validation"], bert_scores(bert, texts["validation"]))
        history.append(dict(epoch=epoch, loss=float(np.mean(losses)), **val))
        if val["macro_f1"] > best:
            best, best_epoch = val["macro_f1"], epoch
            state = copy.deepcopy({k: v.cpu() for k, v in bert.state_dict().items()})
        print("RuBERT epoch", epoch, "val F1", round(val["macro_f1"], 4))
    bert.load_state_dict(state)
    bert.save_pretrained(bert_path, safe_serialization=True)
    tokenizer.save_pretrained(bert_path)
    (RUN_DIR / "bert_history.json").write_text(
        json.dumps(
            dict(
                history=history,
                best_epoch=best_epoch,
                data_hashes=hashes,
                revision=REVISION,
            )
        )
    )
info = json.loads((RUN_DIR / "bert_history.json").read_text(encoding="utf-8"))
assert info["data_hashes"] == hashes and info["revision"] == REVISION
models["RuBERT-tiny2"] = bert
histories["RuBERT-tiny2"] = info["history"]
validation["RuBERT-tiny2"] = metrics(targets["validation"], bert_scores(bert, texts["validation"]))
records["RuBERT-tiny2"] = {
    "epoch": info["best_epoch"],
    "seed": 20261106,
    "revision": REVISION,
}
print("Выбрана эпоха", info["best_epoch"])

## 6. Итоговое сравнение
Сначала фиксируем победителя по валидации, потом рассчитываем Test для каждого классификатора. Новые параметры по тестовым результатам не выбираются.

При равенстве validation macro-F1 выбирается первая модель в порядке сравнения. В этом запуске SVM и RuBERT равны по валидации, поэтому выбран SVM. Результат Test не используется для разрешения равенства.

In [ ]:
winner = max(validation, key=lambda name: validation[name]["macro_f1"])
print("Выбрана по валидации:", winner)


def scores_for(name, batch):
    if name == "TextCNN":
        return cnn_scores(models[name], batch)
    if name == "RuBERT-tiny2":
        return bert_scores(models[name], batch)
    return classic_scores(models[name], batch)


test_scores = {name: scores_for(name, texts["test"]) for name in models}
test_metrics = {name: metrics(targets["test"], s) for name, s in test_scores.items()}
summary = pd.DataFrame(
    [
        dict(
            model=name,
            validation_macro_f1=validation[name]["macro_f1"],
            **test_metrics[name],
        )
        for name in models
    ]
).set_index("model")
display(summary.round(4))
summary.to_csv(RUN_DIR / "comparison.csv")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, scores) in zip(axes.flat, test_scores.items(), strict=False):
    matrix = confusion_matrix(targets["test"], scores.argmax(1), labels=range(3))
    ConfusionMatrixDisplay(matrix, display_labels=RUSSIAN).plot(
        ax=ax, colorbar=False, cmap="Blues", values_format="d"
    )
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=15)
axes.flat[-1].axis("off")
fig.tight_layout()
fig.savefig(RUN_DIR / "confusion_matrices.png", dpi=150)
plt.show()
for name, scores in test_scores.items():
    print(name)
    display(
        pd.DataFrame(
            classification_report(
                targets["test"],
                scores.argmax(1),
                labels=range(3),
                target_names=RUSSIAN,
                output_dict=True,
                zero_division=0,
            )
        ).T.round(4)
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, ax in enumerate(axes):
    for name, scores in test_scores.items():
        y = targets["test"] == k
        precision, recall, _ = precision_recall_curve(y, scores[:, k])
        ap = average_precision_score(y, scores[:, k])
        ax.plot(recall, precision, label=f"{name}: AP={ap:.3f}")
    ax.set(
        title=RUSSIAN[k],
        xlabel="Recall",
        ylabel="Precision",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(RUN_DIR / "precision_recall.png", dpi=150)
plt.show()

## 7. Анализ ошибок
Проверим длину и незнакомую лексику. OOV ниже считается относительно обучающего словаря CNN. Это наблюдательные срезы. Их корреляция с ошибками сама по себе не доказывает причину. Словари TF-IDF и RuBERT отличаются.

In [ ]:
lens = np.array([len(t.split()) for t in texts["test"]])
oov = np.array([sum(w not in vocabulary for w in tokens(t)) / max(len(tokens(t)), 1) for t in texts["test"]])
slices = []
for name, scores in test_scores.items():
    for title, mask in (
        ("до 100 слов", lens <= 100),
        ("более 100 слов", lens > 100),
        ("OOV CNN до 5%", oov <= 0.05),
        ("OOV CNN более 5%", oov > 0.05),
    ):
        if mask.any():
            slices.append(
                dict(
                    model=name,
                    slice=title,
                    n=int(mask.sum()),
                    accuracy=accuracy_score(targets["test"][mask], scores[mask].argmax(1)),
                )
            )
display(pd.DataFrame(slices).round(4))
short = pd.DataFrame(
    [
        dict(
            model=name,
            **metrics(
                targets["test"],
                scores_for(name, [" ".join(t.split()[:40]) for t in texts["test"]]),
            ),
        )
        for name in models
    ]
).set_index("model")
print("Срез первых 40 слов тех же статей")
display(short.round(4))
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for name, history in histories.items():
    axes[0].plot(
        [r["epoch"] for r in history],
        [r["loss"] for r in history],
        marker="o",
        label=name,
    )
    axes[1].plot(
        [r["epoch"] for r in history],
        [r["macro_f1"] for r in history],
        marker="o",
        label=name,
    )
axes[0].set(xlabel="Эпоха", ylabel="Train loss")
axes[1].set(xlabel="Эпоха", ylabel="Validation macro-F1")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
errors = []
for name, scores in test_scores.items():
    for i in np.flatnonzero(scores.argmax(1) != targets["test"]):
        row = splits["test"][i]
        errors.append(
            dict(
                model=name,
                id=row["id"],
                title=row["title"],
                url=row["url"],
                weak_label=row["label"],
                prediction=LABELS[scores[i].argmax()],
                words=int(lens[i]),
                cnn_oov=float(oov[i]),
            )
        )
error_table = pd.DataFrame(errors)
for name in models:
    subset = error_table[error_table["model"] == name]
    print(name, "расхождений", len(subset))
    display(subset[["title", "weak_label", "prediction", "words", "cnn_oov"]].head(5))
error_table.to_csv(RUN_DIR / "errors.csv", index=False)

### Пять конкретных примеров исходного Naive Bayes

| Сюжет | Автоматическая метка | Предсказание | Разбор |
|---|---|---|---|
| Huawei и внедрение ОС «Аврора» | технологии | образование | Тема описана через переговоры. Признаки «встречи», «рабочей группы», «министерство» сдвигают оценку к образованию. |
| Обсуждение проката голливудских фильмов | развлечения | образование | Административные слова «министр», «сократить», «поправки» перевешивают признаки кино. |
| Обязательный просмотр кино в учебной программе | развлечения | образование | Шум метки — содержательно речь об учебной программе. |
| Умные часы для обучения программированию | образование | технологии | Смешение тем — учебная цель, но большая часть текста описывает устройство. |
| Пожар из-за зарядного устройства | технологии | образование | Сюжет о происшествии вне трёх классов. И метка, и предсказание не соответствуют основной теме. |

Слова выделены по вкладу в разность оценок Naive Bayes в отдельном разборе проекта. Ниже сверяются реальные документы и ответы всех моделей.

In [ ]:
prefixes = [
    "49fb9f51ff38",
    "533c8ac0d3b5",
    "95babd80ea81",
    "09a8bc29b6bc",
    "899c5ef3b427",
]
cases = []
for prefix in prefixes:
    found = [i for i, row in enumerate(splits["test"]) if row["id"].startswith(prefix)]
    assert len(found) == 1
    i = found[0]
    cases.append(
        dict(
            title=splits["test"][i]["title"],
            weak_label=splits["test"][i]["label"],
            **{name: LABELS[scores[i].argmax()] for name, scores in test_scores.items()},
        )
    )
display(pd.DataFrame(cases))
for name in ("TextCNN", "RuBERT-tiny2"):
    scores = test_scores[name]
    ids = np.flatnonzero(scores.argmax(1) != targets["test"])
    print("\n", name, "примеры расхождений")
    for i in ids[:3]:
        print("Метка", LABELS[targets["test"][i]], "ответ", LABELS[scores[i].argmax()])
        print(" ".join(texts["test"][i].split()[:60]))

### Ошибки нейросетей

| Модель и сюжет | Метка → ответ | Разбор |
|---|---|---|
| TextCNN — финал «Евровидения» | развлечения → технологии | В тексте есть YouTube, Google и поисковые запросы. Гипотеза — локальные технологические сочетания отвлекли CNN от темы музыкального конкурса. Для подтверждения нужно отдельно убрать эти упоминания и повторить предсказание. |
| TextCNN — скорость загрузки PlayStation 5 | технологии → развлечения | Фрагмент одновременно описывает устройство и видеоигру. В основной части говорится о скорости консоли, а игровые признаки могли сместить оценку к развлечениям. |
| TextCNN — кража с участием актёра | развлечения → образование | Содержательно это происшествие. Ошибочна автоматическая развлекательная метка, а предсказание образования также не соответствует смыслу. |
| RuBERT — школьная программа с обязательным просмотром кино | развлечения → образование | Единственное расхождение RuBERT. Смысловой ответ образования обоснован учебной программой, хотя автоматическая метка опирается на кино. Это пример шума эталона. |

Ошибки локальных признаков CNN и сомнительные метки требуют разных действий: в первом случае полезны новые обучающие примеры/проверки устойчивости, во втором — пересмотр разметки.


### Интерпретация
- Аудит 60 новостей дал 51 чистую, 6 шумных, 3 спорных. Это диагностика подвыборки в один проход без независимого согласования, а не доля качества всего корпуса.
- Пограничные сюжеты пересекают образование, технологии и административные темы. Упоминание университета не гарантирует образовательный смысл.
- Падение train loss при остановке роста validation F1 указывает на риск переобучения; используется лучший чекпоинт.
- Сравнение 40-словных фрагментов проверяет влияние сокращения. Это те же статьи, результаты коррелированы с полным Test.
- Class_weight не варьировали, поскольку Train сбалансирован. Стоп-слова и лемматизация тоже не варьировались, их улучшение не доказано.
- Для объяснения ошибки нейросети выводятся сами входы и классы. Причину по одному тексту можно предположить; причинный вывод требует отдельной проверки.
- Следующий шаг качества — независимые метки и реальные расшифровки. Проблема посторонних тем выходит за трёхклассовую постановку этого ноутбука.

## 8. Выводы и сохранение
Итоговая таблица позволяет выбрать лучший исследовательский вариант. Для переноса в бот нужны проверка на ASR, совместимый адаптер и оценка размера/скорости. Небольшие различия на одном Test не доказывают превосходство на любом потоке сообщений.

In [ ]:
print("Победитель по validation:", winner)
print("Его Test:", test_metrics[winner])
for name in ("TextCNN", "RuBERT-tiny2"):
    delta = test_metrics[name]["macro_f1"] - test_metrics["MultinomialNB"]["macro_f1"]
    print(name, "разность test macro-F1 с Naive Bayes", round(delta, 4))
result = dict(
    labels=LABELS,
    versions=VERSIONS,
    device=str(DEVICE),
    data_hashes=hashes,
    selection="maximum validation macro-F1",
    selected_model=winner,
    validation=validation,
    test=test_metrics,
    short_40_words=short.to_dict(orient="index"),
    training=records,
    histories=histories,
    confusion_matrices={
        name: confusion_matrix(targets["test"], s.argmax(1), labels=range(3)).tolist()
        for name, s in test_scores.items()
    },
    label_status="preliminary automatic labels",
    pr_auc="macro AP one-vs-rest",
)
(RUN_DIR / "nlp_results.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
np.savez_compressed(RUN_DIR / "test_scores.npz", y=targets["test"], **test_scores)
for name in estimators:
    joblib.dump(models[name], RUN_DIR / f"{name}.joblib", compress=3)
for name, s in test_scores.items():
    assert metrics(targets["test"], s) == test_metrics[name]
print("Сохранено в", RUN_DIR)